# chatbot evaluation

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["LANGSMITH_API_KEY"] = os.getenv("LANGSMITH_API_KEY")
os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")
os.environ["LANGSMITH_TRACING"] = "true"

In [5]:
from langsmith import Client

client = Client()

# Create dataset
dataset_name = "Evaluation using LangSmith"

dataset = client.create_dataset(
    dataset_name=dataset_name
)

# Add examples one by one
client.create_example(
    dataset_id=dataset.id,
    inputs={
        "question": "What is LangChain?"
    },
    outputs={
        "answer": "LangChain is a framework for building applications powered by large language models."
    }
)

client.create_example(
    dataset_id=dataset.id,
    inputs={
        "question": "What is LangSmith?"
    },
    outputs={
        "answer": "LangSmith is a platform for tracing, evaluating, testing, and monitoring LLM applications."
    }
)

client.create_example(
    dataset_id=dataset.id,
    inputs={
        "question": "What is RAG?"
    },
    outputs={
        "answer": "RAG stands for Retrieval-Augmented Generation. It retrieves relevant information from external sources and provides it to an LLM to generate an answer."
    }
)

client.create_example(
    dataset_id=dataset.id,
    inputs={
        "question": "What are embeddings?"
    },
    outputs={
        "answer": "Embeddings are numerical representations of data that capture semantic meaning and can be used for similarity search."
    }
)

client.create_example(
    dataset_id=dataset.id,
    inputs={
        "question": "What is a vector database?"
    },
    outputs={
        "answer": "A vector database stores and searches vector embeddings efficiently, which makes it useful for semantic search and RAG applications."
    }
)

print("Dataset and examples created successfully!")

Dataset and examples created successfully!


In [ ]:
import os
from langchain_google_genai import ChatGoogleGenerativeAI
os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")

# 2. Create Gemini Judge Model
judge_llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0
)

# 3. Evaluation instructions
eval_instructions = """
You are an expert professor specialized in grading
student answers.

Your task is to compare a student's predicted answer
with the reference answer.

Determine whether the student's answer is factually
correct.

Respond with ONLY one word:

CORRECT

or

INCORRECT
"""


# 4. Correctness evaluator

def correctness(
    inputs: dict,
    outputs: dict,
    reference_outputs: dict
) -> bool:

    user_content = f"""
Question:
{inputs["question"]}

Reference Answer:
{reference_outputs["answer"]}

Predicted Answer:
{outputs["response"]}

Is the predicted answer factually correct
compared with the reference answer?

Respond with only:
CORRECT
or
INCORRECT
"""

    response = judge_llm.invoke([
        (
            "system",
            eval_instructions
        ),
        (
            "human",
            user_content
        )
    ])

    result = response.content.strip().upper()
    return result == "CORRECT"

In [8]:
## Concisions : checks whether the acutal output is less than 2x the length of the excepted result.

def concision(outputs: dict, reference_outputs: dict) -> bool:
    return int(len(outputs["response"]) < 2 * len(reference_outputs["answer"]))

## Run Evaluation

In [9]:
default_instructions = "Respond to the user's question in a short, concise manner (one short sentence)."


def my_app(
    question: str,
    model: str = "gemini-2.5-flash",
    instructions: str = default_instructions
) -> dict:

    response = judge_llm.invoke([
        ("system", instructions),
        ("human", question)
    ])

    return {
        "response": response.content
    }

In [12]:
# Call my_app for every data point
def ls_target(inputs: dict) -> dict:
    return my_app(inputs["question"])


In [21]:

# ============================================================
# LangSmith Evaluation with Gemini
# ============================================================

import os
from dotenv import load_dotenv

from langsmith import Client
from langchain_google_genai import ChatGoogleGenerativeAI


# ============================================================
# 1. LOAD ENVIRONMENT VARIABLES
# ============================================================

load_dotenv(override=True)

LANGSMITH_API_KEY = os.getenv("LANGSMITH_API_KEY")
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

if not GOOGLE_API_KEY:
    raise ValueError(
        "GOOGLE_API_KEY is missing. "
        "Add GOOGLE_API_KEY=your_key to your .env file."
    )

if not LANGSMITH_API_KEY:
    raise ValueError(
        "LANGSMITH_API_KEY is missing. "
        "Add LANGSMITH_API_KEY=your_key to your .env file."
    )


# ============================================================
# 2. LANGSMITH CONFIGURATION
# ============================================================

os.environ["LANGSMITH_API_KEY"] = LANGSMITH_API_KEY
os.environ["LANGSMITH_TRACING"] = "true"


# ============================================================
# 3. INITIALIZE LANGSMITH CLIENT
# ============================================================

client = Client()


# ============================================================
# 4. INITIALIZE GEMINI
# ============================================================

MODEL_NAME = "gemini-3.5-flash-lite"

gemini_llm = ChatGoogleGenerativeAI(
    model=MODEL_NAME,
    google_api_key=GOOGLE_API_KEY
)


# ============================================================
# 5. HELPER FUNCTION
# ============================================================

def extract_text(response) -> str:
    """
    Extract text safely from a LangChain AIMessage.

    Depending on the langchain-google-genai version,
    response.content can be:

        "plain string"

    OR:

        [
            {
                "type": "text",
                "text": "some response"
            }
        ]
    """

    content = response.content

    # --------------------------------------------------------
    # Case 1: content is already a string
    # --------------------------------------------------------

    if isinstance(content, str):
        return content.strip()

    # --------------------------------------------------------
    # Case 2: content is a list of content blocks
    # --------------------------------------------------------

    if isinstance(content, list):

        text_parts = []

        for block in content:

            # Example:
            # {
            #     "type": "text",
            #     "text": "Hello"
            # }

            if isinstance(block, dict):

                if block.get("type") == "text":

                    text = block.get("text", "")

                    if text:
                        text_parts.append(str(text))

            # Some versions may return objects instead
            # of dictionaries.

            elif hasattr(block, "text"):

                text = getattr(block, "text", "")

                if text:
                    text_parts.append(str(text))

        return "\n".join(text_parts).strip()

    # --------------------------------------------------------
    # Fallback
    # --------------------------------------------------------

    return str(content).strip()


# ============================================================
# 6. TEST GEMINI CONNECTION
# ============================================================

print("\nTesting Gemini API...")

test_response = gemini_llm.invoke(
    "What is LangChain? Answer in one short sentence."
)

test_text = extract_text(test_response)

print("\nGemini response:")
print(test_text)


# ============================================================
# 7. DATASET
# ============================================================

dataset_name = "langchain-evaluation-gemini"


# ============================================================
# 8. CREATE DATASET IF IT DOES NOT EXIST
# ============================================================

try:

    dataset = client.read_dataset(
        dataset_name=dataset_name
    )

    print(f"\nDataset already exists: {dataset_name}")

except Exception:

    print(f"\nCreating dataset: {dataset_name}")

    dataset = client.create_dataset(
        dataset_name=dataset_name,
        description=(
            "Dataset for evaluating Gemini responses "
            "to basic LangChain questions."
        )
    )

    examples = [
        {
            "question": "What is LangChain?",
            "answer": (
                "LangChain is a framework for building "
                "applications powered by language models."
            )
        },
        {
            "question": "What is RAG?",
            "answer": (
                "RAG stands for Retrieval-Augmented Generation "
                "and combines document retrieval with language generation."
            )
        },
        {
            "question": "What is an AI agent?",
            "answer": (
                "An AI agent is a system that can reason, "
                "use tools, and take actions to accomplish a task."
            )
        },
        {
            "question": "What is LangGraph?",
            "answer": (
                "LangGraph is a framework for building "
                "stateful and controllable agent workflows."
            )
        },
        {
            "question": "What is a vector database?",
            "answer": (
                "A vector database stores embeddings and "
                "enables similarity search over vector representations."
            )
        }
    ]

    for example in examples:

        client.create_example(
            inputs={
                "question": example["question"]
            },
            outputs={
                "answer": example["answer"]
            },
            dataset_id=dataset.id
        )

    print("Dataset created successfully.")


# ============================================================
# 9. TARGET APPLICATION
# ============================================================

DEFAULT_INSTRUCTIONS = (
    "Answer the user's question in one short, "
    "clear and concise sentence."
)


def my_app(
    question: str,
    instructions: str = DEFAULT_INSTRUCTIONS
) -> dict:

    response = gemini_llm.invoke(
        [
            ("system", instructions),
            ("human", question)
        ]
    )

    response_text = extract_text(response)

    return {
        "response": response_text
    }


# ============================================================
# 10. LANGSMITH TARGET FUNCTION
# ============================================================

def ls_target(inputs: dict) -> dict:

    question = inputs["question"]

    return my_app(
        question=question
    )


# ============================================================
# 11. CORRECTNESS EVALUATOR
# ============================================================

eval_instructions = """
You are an expert professor evaluating student answers.

Compare the predicted answer with the reference answer.

The predicted answer does NOT need to use exactly
the same wording as the reference answer.

Judge whether it is factually correct and answers
the question appropriately.

Return ONLY one of:

CORRECT
INCORRECT

Do not provide explanations.
"""


def correctness(
    inputs: dict,
    outputs: dict,
    reference_outputs: dict
) -> bool:

    question = inputs["question"]

    reference_answer = reference_outputs["answer"]

    predicted_answer = outputs["response"]

    user_content = f"""
Question:
{question}

Reference Answer:
{reference_answer}

Predicted Answer:
{predicted_answer}

Is the predicted answer factually correct
and appropriate according to the reference answer?

Return ONLY:

CORRECT

or

INCORRECT
"""

    response = gemini_llm.invoke(
        [
            ("system", eval_instructions),
            ("human", user_content)
        ]
    )

    # IMPORTANT:
    # Gemini may return response.content as a list.
    result = extract_text(response).upper()

    # Extra protection if Gemini returns something like:
    # "CORRECT."
    result = result.replace(".", "").strip()

    return result == "CORRECT"


# ============================================================
# 12. CONCISION EVALUATOR
# ============================================================

concision_instructions = """
You are an expert evaluator.

Determine whether the answer is concise.

The answer should:

1. Directly answer the question.
2. Avoid unnecessary explanation.
3. Avoid repetition.
4. Be short and clear.

Return ONLY:

YES

or

NO

Do not provide explanations.
"""


def concision(
    inputs: dict,
    outputs: dict,
    reference_outputs: dict
) -> bool:

    question = inputs["question"]

    predicted_answer = outputs["response"]

    user_content = f"""
Question:
{question}

Answer:
{predicted_answer}

Is this answer concise and directly relevant
to the question?

Return ONLY:

YES

or

NO
"""

    response = gemini_llm.invoke(
        [
            ("system", concision_instructions),
            ("human", user_content)
        ]
    )

    # IMPORTANT:
    # Gemini may return a list instead of a string.
    result = extract_text(response).upper()

    # Extra protection if Gemini returns:
    # "YES."
    result = result.replace(".", "").strip()

    return result == "YES"


# ============================================================
# 13. RUN LANGSMITH EVALUATION
# ============================================================

print("\n========================================")
print("Running LangSmith evaluation...")
print("========================================\n")


experiment_results = client.evaluate(
    ls_target,
    data=dataset_name,
    evaluators=[
        correctness,
        concision
    ],
    experiment_prefix="gemini-3.5-flash-lite-evaluation"
)


# ============================================================
# 14. FINISHED
# ============================================================

print("\n========================================")
print("Evaluation completed successfully!")
print("========================================")

print("\nOpen LangSmith to view:")

print("- Dataset")
print("- Experiments")
print("- Predictions")
print("- Correctness scores")
print("- Concision scores")




Testing Gemini API...

Gemini response:
LangChain is a framework that simplifies building applications with large language models (LLMs) by chaining together various components like data sources, prompts, and APIs.

Dataset already exists: langchain-evaluation-gemini

Running LangSmith evaluation...

View the evaluation results for experiment: 'gemini-3.5-flash-lite-evaluation-e88a74b4' at:
https://smith.langchain.com/o/1dedea63-72fd-48b3-b04c-058788302da7/datasets/d7db1319-23b3-450d-a288-d5cdfe31078a/compare?selectedSessions=fbdb1ba5-aaa0-41f0-9ae8-26240234789c




5it [00:18,  3.76s/it]


Evaluation completed successfully!

Open LangSmith to view:
- Dataset
- Experiments
- Predictions
- Correctness scores
- Concision scores


In [16]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)

google_key = os.getenv("GOOGLE_API_KEY")

print("Key exists:", google_key is not None)
print("Key length:", len(google_key) if google_key else 0)
print("Key prefix:", google_key[:10] if google_key else None)

Key exists: True
Key length: 53
Key prefix: AQ.Ab8RN6I


In [18]:
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI

load_dotenv(override=True)

GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

if not GOOGLE_API_KEY:
    raise ValueError("GOOGLE_API_KEY is missing")

gemini = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite",
    google_api_key=GOOGLE_API_KEY,
    temperature=0
)

response = gemini.invoke(
    "What is LangChain? Answer in one short sentence."
)

print(response.content)

/home/dhinesh/Ai-engineer/venv/lib/python3.12/site-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


[{'type': 'text', 'text': 'LangChain is a framework designed to simplify the creation of applications using large language models by chaining together various components like data sources, prompts, and APIs.', 'extras': {'signature': 'El4KXAFpFH0T0Rked7Tia+P8zVEPZvho34FNZLcQq0rMKVaygs9LISjN/b9V8GvvH/E2ZpUbBrjUdyr+zdyGo744ie/kYcYWHDn6VuYaI4JpkxXMtNcr3QWvRTK23mLN'}}]


## Evaluation of RAG

In [24]:
# ============================================================
# RAG - Gemini Embeddings with Batch Control
# ============================================================

import os
import time
from dotenv import load_dotenv

from langchain_community.document_loaders import WebBaseLoader
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

# ============================================================
# 1. Load API Key
# ============================================================

load_dotenv(override=True)

GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

if not GOOGLE_API_KEY:
    raise ValueError("GOOGLE_API_KEY is not set in your .env file")

# ============================================================
# 2. URLs
# ============================================================

urls = [
    "https://lilianweng.github.io/posts/2023-06-23-agent/",
    "https://lilianweng.github.io/posts/2023-03-15-prompt-engineering/",
    "https://lilianweng.github.io/posts/2023-10-25-adv-attack-llm/",
]

# ============================================================
# 3. Load documents
# ============================================================

docs = [WebBaseLoader(url).load() for url in urls]

docs_list = [
    item
    for sublist in docs
    for item in sublist
]

print(f"Loaded documents: {len(docs_list)}")

# ============================================================
# 4. Split documents
# ============================================================

text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=500,
    chunk_overlap=50,
)

doc_splits = text_splitter.split_documents(docs_list)

print(f"Total chunks: {len(doc_splits)}")

# ============================================================
# 5. Gemini Embeddings
# ============================================================

embeddings = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-001",
    google_api_key=GOOGLE_API_KEY,
)

# ============================================================
# 6. Create Vector Store
# ============================================================
# Gemini free tier has request limits.
# Process documents in smaller batches instead of sending
# everything at once.

vectorstore = InMemoryVectorStore(embeddings)

BATCH_SIZE = 20

for i in range(0, len(doc_splits), BATCH_SIZE):

    batch = doc_splits[i:i + BATCH_SIZE]

    print(
        f"Embedding chunks {i + 1} - "
        f"{min(i + BATCH_SIZE, len(doc_splits))} "
        f"of {len(doc_splits)}"
    )

    try:
        vectorstore.add_documents(batch)

    except Exception as e:

        if "429" in str(e) or "RESOURCE_EXHAUSTED" in str(e):

            print("Rate limit reached. Waiting 50 seconds...")
            time.sleep(50)

            vectorstore.add_documents(batch)

        else:
            raise e

    # Small delay between batches
    time.sleep(2)

# ============================================================
# 7. Create Retriever
# ============================================================

retriever = vectorstore.as_retriever(
    search_kwargs={"k": 6}
)

# ============================================================
# 8. Test Retrieval
# ============================================================

query = "What is an AI agent?"

results = retriever.invoke(query)

print("\nRetrieved Documents:")
print("=" * 60)

for i, doc in enumerate(results, start=1):

    print(f"\n--- Result {i} ---")
    print(doc.page_content[:1000])

Loaded documents: 3
Total chunks: 91
Embedding chunks 1 - 20 of 91
Embedding chunks 21 - 40 of 91
Embedding chunks 41 - 60 of 91
Embedding chunks 61 - 80 of 91
Embedding chunks 81 - 91 of 91

Retrieved Documents:

--- Result 1 ---
Each element is an observation, an event directly provided by the agent.
- Inter-agent communication can trigger new natural language statements.


Retrieval model: surfaces the context to inform the agent’s behavior, according to relevance, recency and importance.

Recency: recent events have higher scores
Importance: distinguish mundane from core memories. Ask LM directly.
Relevance: based on how related it is to the current situation / query.


Reflection mechanism: synthesizes memories into higher level inferences over time and guides the agent’s future behavior. They are higher-level summaries of past events (<- note that this is a bit different from self-reflection above)

Prompt LM with 100 most recent observations and to generate 3 most salient high-l

In [25]:
retriever.invoke("what is agents?")

[Document(id='ee465cfa-77c8-4795-aabf-d1ccb7b1d547', metadata={'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/', 'title': "LLM Powered Autonomous Agents | Lil'Log", 'description': 'Building agents with LLM (large language model) as its core controller is a cool concept. Several proof-of-concepts demos, such as AutoGPT, GPT-Engineer and BabyAGI, serve as inspiring examples. The potentiality of LLM extends beyond generating well-written copies, stories, essays and programs; it can be framed as a powerful general problem solver.\nAgent System Overview\nIn a LLM-powered autonomous agent system, LLM functions as the agent’s brain, complemented by several key components:\n\nPlanning\n\nSubgoal and decomposition: The agent breaks down large tasks into smaller, manageable subgoals, enabling efficient handling of complex tasks.\nReflection and refinement: The agent can do self-criticism and self-reflection over past actions, learn from mistakes and refine them for future steps, 

In [27]:
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

load_dotenv(override=True)

llm = init_chat_model(
    "gemini-2.5-flash-lite",
    model_provider="google_genai",
    api_key=os.getenv("GOOGLE_API_KEY"),
)

llm

ChatGoogleGenerativeAI(metadata={'lc_versions': {'langchain-core': '1.6.3', 'langchain': '1.4.2', 'langchain-google-genai': '4.4.0'}}, profile={'name': 'Gemini 2.5 Flash-Lite', 'release_date': '2025-06-17', 'last_updated': '2025-06-17', 'open_weights': False, 'max_input_tokens': 1048576, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': True, 'pdf_inputs': True, 'video_inputs': True, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'image_tool_message': True, 'tool_choice': True}, google_api_key=SecretStr('**********'), model='gemini-2.5-flash-lite', client=<google.genai.client.Client object at 0x71c5d75fb140>, default_metadata=(), model_kwargs={})

In [28]:
from langsmith import traceable

## ADd decorator
@traceable
def rag_bot(question:str) -> dict:
    docs = retriever.invoke(question)
    docs_string = " ".join(doc.page_content for doc in docs)
    instructions = f"""You are a helpful assistant who is good at analyzing source inforamtion and answer the question in concise manner. use three simple bullet points for answer
    Documents: 
        {docs_string}
    """
    ## llm invoke
    response = llm.invoke([
        {"role":"system","content":instructions},
        {"role":"user","content":question}
    ])
    return {"answer": response.content,"documents":docs}



In [29]:
rag_bot("what is agents")

{'answer': 'Here\'s a breakdown of what agents are in the context of LLM-powered systems:\n\n*   **LLM-controlled entities:** Agents are systems where a Large Language Model (LLM) acts as the core controller or "brain."\n*   **Equipped with components:** They are enhanced with planning, memory, and tool-use capabilities to perform tasks.\n*   **Autonomous operation:** Agents are designed to make decisions and act independently without constant user intervention, pursuing defined goals.',
 'documents': [Document(id='ee465cfa-77c8-4795-aabf-d1ccb7b1d547', metadata={'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/', 'title': "LLM Powered Autonomous Agents | Lil'Log", 'description': 'Building agents with LLM (large language model) as its core controller is a cool concept. Several proof-of-concepts demos, such as AutoGPT, GPT-Engineer and BabyAGI, serve as inspiring examples. The potentiality of LLM extends beyond generating well-written copies, stories, essays and programs; 

# Dataset

In [ ]:
from langsmith import Client

client = Client()

examples = [
    {
        "inputs": {
            "question": "What is an AI agent?"
        },
        "outputs": {
            "answer": "An AI agent is a system that can perceive a task, reason about what to do, use tools when necessary, and take actions to achieve a goal."
        }
    },

    {
        "inputs": {
            "question": "What is the ReAct pattern in AI agents?"
        },
        "outputs": {
            "answer": "ReAct combines reasoning and action. The agent reasons about the current task, selects an action or tool, observes the result, and continues until it can produce a final answer."
        }
    },

    {
        "inputs": {
            "question": "Why do AI agents need tools?"
        },
        "outputs": {
            "answer": "Tools allow agents to perform tasks that an LLM cannot reliably do by itself, such as searching the web, querying databases, executing code, calling APIs, or accessing external systems."
        }
    },

    {
        "inputs": {
            "question": "What is tool calling in an AI agent?"
        },
        "outputs": {
            "answer": "Tool calling allows an LLM to request the execution of a predefined function with structured arguments. The tool executes the operation and returns its result to the model."
        }
    },
      {
        "inputs": {
            "question": "What is the difference between an AI agent and a chatbot?"
        },
        "outputs": {
            "answer": "A chatbot primarily generates conversational responses, while an AI agent can reason about a goal, use tools, maintain state, and perform actions to complete tasks."
        }
    },
    {
        "inputs": {
            "question": "What is RAG?"
        },
        "outputs": {
            "answer": "RAG, or Retrieval-Augmented Generation, retrieves relevant information from an external knowledge source and provides it to an LLM as context before generating an answer."
        }
    },

    {
        "inputs": {
            "question": "What are the main steps in a RAG pipeline?"
        },
        "outputs": {
            "answer": "A typical RAG pipeline loads documents, splits them into chunks, creates embeddings, stores them in a vector database, retrieves relevant chunks for a query, and passes the retrieved context to an LLM."
        }
    },

    {
        "inputs": {
            "question": "What are embeddings in RAG?"
        },
        "outputs": {
            "answer": "Embeddings are numerical vector representations of text that capture semantic meaning. They allow a vector store to find documents that are semantically similar to a user's query."
        }
    },
]

### Create the dataset and examples in LAngsmith

dataset_name = "RAG test Evaluation"
dataset = client.create_dataset(dataset_name=dataset_name)
client.create_examples(
    dataset_id=dataset.id,
    examples=examples
)


{'example_ids': ['36b580eb-6809-42d9-91af-f06fe33413d0',
  '717b0651-398f-43d3-af4d-00aaaf9b5714',
  '5eafe7f5-1336-48b0-a208-196a057b811d',
  '4ed494d9-fa56-478c-a5e1-57707b8d5e0f',
  'ab4ec2d4-2700-401a-93a8-4740f0792b25',
  '3d7eaecf-da85-4e9f-b022-5494b5f7d0bc',
  'b2a91cfe-72d1-420e-8605-b009b90643fe',
  '4fdfcb3f-ce94-4937-969f-a5982ea353c0'],
 'count': 8,
 'as_of': '2026-09-20T11:19:32.700294104Z'}

# Evaluators

In [33]:
from typing_extensions import Annotated, TypedDict # Correctness output schema

class CorrectnessGrade(TypedDict): 
    explanation: Annotated[str, "Explain you reasoning for the score"] 
    correct:Annotated[bool,"True if the answer is correct, False Otherwise"]
    
correctness_instructions = """
You are an expert teacher grading a quiz answer.

You will be given:
- QUESTION: The question asked to the student.
- GROUND TRUTH: The correct reference answer.
- STUDENT ANSWER: The answer provided by the student.

Grade the student's answer using ONLY factual accuracy relative to the ground truth.

Evaluation criteria:
1. Check whether the student answer correctly addresses the question.
2. Compare the factual claims in the student answer with the ground truth.
3. The student answer does not need to use the same wording as the ground truth.
4. The student answer may contain additional information, as long as that information is factually correct and does not conflict with the ground truth.
5. If the student answer contains a factual contradiction, incorrect claim, or misleading statement that affects the answer, mark it as incorrect.
6. If the student answer is incomplete and misses an important part of the ground truth, mark it as incorrect.
7. Do not penalize differences in wording, style, formatting, or level of detail.
8. Do not use outside knowledge to override the ground truth. Evaluate the answer primarily against the provided ground truth.
9. Do not give partial credit. The final result must be either True or False.

Correctness:
- correct = True if the student's answer satisfies all the criteria above.
- correct = False if the student's answer fails any important criterion.

For the explanation:
- Explain the reasoning clearly and step by step.
- Identify the important claims in the student's answer.
- Compare them with the ground truth.
- Point out any missing, incorrect, or conflicting information.
- Do not simply state the correct answer at the beginning.
- Do not judge the student's writing style or grammar unless it changes the factual meaning.

Return only the structured output matching the provided schema.
"""



In [35]:
import os

from langchain.chat_models import init_chat_model


grader_llm = init_chat_model(
    "gemini-2.5-flash-lite",
    model_provider="google_genai",
    api_key=os.getenv("GOOGLE_API_KEY"),
).with_structured_output(
    CorrectnessGrade,
    method="json_schema",
)


def correctness(
    inputs: dict,
    outputs: dict,
    reference_outputs: dict,
) -> bool:
    """
    Evaluator for RAG answer factual accuracy.
    """

    answers = f"""
QUESTION:
{inputs["question"]}

GROUND TRUTH ANSWER:
{reference_outputs["answer"]}

STUDENT ANSWER:
{outputs["answer"]}
"""

    grade = grader_llm.invoke(
        [
            {
                "role": "system",
                "content": correctness_instructions,
            },
            {
                "role": "user",
                "content": answers,
            },
        ]
    )

    return grade["correct"]

# Relevance

In [36]:
from typing_extensions import Annotated, TypedDict


class RelevanceGrade(TypedDict):
    explanation: Annotated[
        str,
        "Explain why the answer is or is not relevant to the question"
    ]
    relevant: Annotated[
        bool,
        "True if the answer is relevant to the question, False otherwise"
    ]


relevance_instructions = """
You are an expert teacher evaluating the relevance of an answer.

You will be given:
- QUESTION: The question asked by the user.
- ANSWER: The answer generated by the AI system.

Evaluate ONLY whether the answer is relevant to the question.

Evaluation criteria:
1. The answer should directly address the question.
2. The answer should contain information related to the user's question.
3. Do not require the answer to use the same wording as the question.
4. Additional information is acceptable if it remains relevant.
5. If the answer is mostly unrelated, evasive, or fails to address the question, mark it as not relevant.
6. Do not judge factual correctness. Evaluate relevance only.
7. Do not judge grammar, writing style, or formatting.
8. Do not penalize a concise answer if it directly addresses the question.

Relevance:
- relevant = True if the answer directly addresses the question.
- relevant = False if the answer does not meaningfully address the question.

For the explanation:
- Explain why the answer is or is not relevant.
- Identify the connection between the question and answer.
- Do not provide a corrected answer.

Return only the structured output matching the provided schema.
"""


relevance_llm = init_chat_model(
    "gemini-2.5-flash-lite",
    model_provider="google_genai",
    api_key=os.getenv("GOOGLE_API_KEY"),
).with_structured_output(
    RelevanceGrade,
    method="json_schema",
)


def relevance(
    inputs: dict,
    outputs: dict,
) -> bool:
    """
    Evaluator for RAG answer relevance.
    """

    evaluation_input = f"""
QUESTION:
{inputs["question"]}

ANSWER:
{outputs["answer"]}
"""

    grade = relevance_llm.invoke(
        [
            {
                "role": "system",
                "content": relevance_instructions,
            },
            {
                "role": "user",
                "content": evaluation_input,
            },
        ]
    )

    return grade["relevant"]

In [37]:
from typing_extensions import Annotated, TypedDict
from langchain.chat_models import init_chat_model


class GroundednessGrade(TypedDict):
    explanation: Annotated[
        str,
        "Explain whether the answer is supported by the provided context"
    ]
    grounded: Annotated[
        bool,
        "True if the answer is fully supported by the provided context, False otherwise"
    ]


groundedness_instructions = """
You are an expert evaluator checking the groundedness of an AI-generated answer.

You will be given:
- QUESTION: The user's question.
- CONTEXT: The retrieved information provided to the AI system.
- ANSWER: The AI-generated answer.

Evaluate ONLY whether the answer is grounded in the provided context.

Evaluation criteria:
1. Every important factual claim in the answer should be supported by the provided context.
2. Do not assume information that is not present in the context.
3. If the answer contains unsupported factual claims, mark it as not grounded.
4. If the answer contradicts the provided context, mark it as not grounded.
5. The answer may summarize or rephrase the context as long as the meaning is preserved.
6. The answer may contain minor wording differences that do not change the meaning.
7. Do not use outside knowledge to determine whether a claim is supported.
8. Do not judge the writing style, grammar, or relevance.
9. Do not judge whether the context itself is factually correct.
10. If the answer contains multiple important claims and any important claim is unsupported or contradicted by the context, mark it as False.

Groundedness:
- grounded = True if the answer is fully supported by the provided context.
- grounded = False if the answer contains important unsupported or conflicting information.

For the explanation:
- Identify the important claims made in the answer.
- Explain which claims are supported by the context.
- Identify any unsupported or conflicting claims.
- Do not provide a corrected answer.

Return only the structured output matching the provided schema.
"""


groundedness_llm = init_chat_model(
    "gemini-2.5-flash-lite",
    model_provider="google_genai",
    api_key=os.getenv("GOOGLE_API_KEY"),
).with_structured_output(
    GroundednessGrade,
    method="json_schema",
)


def groundedness(
    inputs: dict,
    outputs: dict,
) -> bool:
    """
    Evaluator for RAG answer groundedness.
    """

    context = inputs.get("context", "")

    evaluation_input = f"""
QUESTION:
{inputs["question"]}

CONTEXT:
{context}

ANSWER:
{outputs["answer"]}
"""

    grade = groundedness_llm.invoke(
        [
            {
                "role": "system",
                "content": groundedness_instructions,
            },
            {
                "role": "user",
                "content": evaluation_input,
            },
        ]
    )

    return grade["grounded"]

## Retrieval relevance

In [38]:
from typing_extensions import Annotated, TypedDict
from langchain.chat_models import init_chat_model


class RetrievalRelevanceGrade(TypedDict):
    explanation: Annotated[
        str,
        "Explain whether the retrieved documents are relevant to the question"
    ]
    relevant: Annotated[
        bool,
        "True if the retrieved documents are relevant to the question, False otherwise"
    ]


retrieval_relevance_instructions = """
You are an expert evaluator for a Retrieval-Augmented Generation (RAG) system.

You will be given:
- QUESTION: The user's question.
- RETRIEVED DOCUMENTS: The documents or document chunks retrieved by the RAG system.

Evaluate ONLY the relevance of the retrieved documents to the question.

Evaluation criteria:
1. The retrieved documents should contain information that can help answer the question.
2. The retrieved documents do not need to contain the complete answer.
3. The retrieved documents should be topically and semantically related to the question.
4. Relevant information may be expressed using different wording from the question.
5. If the retrieved documents are unrelated to the question, mark them as not relevant.
6. If the documents contain mostly irrelevant information and do not provide useful evidence for answering the question, mark them as not relevant.
7. Do not judge whether the retrieved information is factually correct.
8. Do not judge the quality of the final answer.
9. Do not use outside knowledge to determine relevance.
10. Focus only on whether the retrieved documents provide useful information for answering the question.

Retrieval Relevance:
- relevant = True if the retrieved documents contain useful information for answering the question.
- relevant = False if the retrieved documents are unrelated or provide no meaningful information for answering the question.

For the explanation:
- Explain the connection between the question and retrieved documents.
- Identify the relevant information found in the retrieved documents.
- If the documents are irrelevant, explain why.
- Do not provide a corrected answer to the question.

Return only the structured output matching the provided schema.
"""


retrieval_relevance_llm = init_chat_model(
    "gemini-2.5-flash-lite",
    model_provider="google_genai",
    api_key=os.getenv("GOOGLE_API_KEY"),
).with_structured_output(
    RetrievalRelevanceGrade,
    method="json_schema",
)


def retrieval_relevance(
    inputs: dict,
    outputs: dict,
) -> bool:
    """
    Evaluator for RAG retrieval relevance.
    """

    retrieved_documents = inputs.get(
        "context",
        inputs.get("documents", "")
    )

    evaluation_input = f"""
QUESTION:
{inputs["question"]}

RETRIEVED DOCUMENTS:
{retrieved_documents}
"""

    grade = retrieval_relevance_llm.invoke(
        [
            {
                "role": "system",
                "content": retrieval_relevance_instructions,
            },
            {
                "role": "user",
                "content": evaluation_input,
            },
        ]
    )

    return grade["relevant"]

# Run the evaluation

In [39]:
def target(inputs: dict) -> dict:
    return rag_bot(inputs["question"])
import pandas

experiment_results = client.evaluate(
    target,
    data=dataset_name,
    evaluators=[correctness,groundedness,relevance,retrieval_relevance],
    experiment_prefix="rag-doc-relevance",
    metadata={"version":"LCEL context, gemini-3.5-flash-lite-preview"},
)
experiment_results.to_pandas()

View the evaluation results for experiment: 'rag-doc-relevance-8017fd39' at:
https://smith.langchain.com/o/1dedea63-72fd-48b3-b04c-058788302da7/datasets/aba52c3e-c6a7-479f-99a3-82e629b46025/compare?selectedSessions=3bb5a239-98a2-4912-bf21-ce05a8bfac8c




0it [00:00, ?it/s]Error running target function: 'dict' object has no attribute 'content'
Traceback (most recent call last):
  File "/home/dhinesh/Ai-engineer/venv/lib/python3.12/site-packages/langsmith/evaluation/_runner.py", line 2052, in _forward
    fn(*args, langsmith_extra=langsmith_extra)
  File "/tmp/ipykernel_7721/2742076871.py", line 2, in target
    return rag_bot(inputs["question"])
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_7721/2892665497.py", line 17, in rag_bot
    return {"answer": response.content,"documents":docs}
                      ^^^^^^^^^^^^^^^^
AttributeError: 'dict' object has no attribute 'content'
Error running evaluator <DynamicRunEvaluator correctness> on run 01a0beaa-9bbd-7322-98f3-7a36f8d37f00: KeyError('answer')
Traceback (most recent call last):
  File "/home/dhinesh/Ai-engineer/venv/lib/python3.12/site-packages/langsmith/evaluation/_runner.py", line 1712, in _run_evaluators
    evaluator_response = evaluator.evaluate_run(  # type:

,inputs.question,outputs.output,error,reference.answer,feedback.correctness,feedback.groundedness,feedback.relevance,feedback.retrieval_relevance,execution_time,example_id,id
0,What are the main steps in a RAG pipeline?,None,"AttributeError(""'dict' object has no attribute...","A typical RAG pipeline loads documents, splits...",None,None,None,False,3.122994,36b580eb-6809-42d9-91af-f06fe33413d0,01a0beaa-9bbd-7322-98f3-7a36f8d37f00
1,What is tool calling in an AI agent?,None,"AttributeError(""'dict' object has no attribute...",Tool calling allows an LLM to request the exec...,None,None,None,False,2.970148,3d7eaecf-da85-4e9f-b022-5494b5f7d0bc,01a0beaa-afa1-7211-bd81-27f16b139498
2,What is the ReAct pattern in AI agents?,None,"AttributeError(""'dict' object has no attribute...",ReAct combines reasoning and action. The agent...,None,None,None,False,2.665360,4ed494d9-fa56-478c-a5e1-57707b8d5e0f,01a0beaa-c2cd-7830-8f99-f8b03ece9100
3,What is RAG?,None,"AttributeError(""'dict' object has no attribute...","RAG, or Retrieval-Augmented Generation, retrie...",None,None,None,False,2.860965,4fdfcb3f-ce94-4937-969f-a5982ea353c0,01a0beaa-d46d-7033-bc5c-2387ccaa8a53
4,What is an AI agent?,None,"AttributeError(""'dict' object has no attribute...",An AI agent is a system that can perceive a ta...,None,None,None,False,2.576578,5eafe7f5-1336-48b0-a208-196a057b811d,01a0beaa-e534-7023-8695-9804ea79c731
5,What are embeddings in RAG?,None,"AttributeError(""'dict' object has no attribute...",Embeddings are numerical vector representation...,None,None,None,False,2.553815,717b0651-398f-43d3-af4d-00aaaf9b5714,01a0beaa-f4d0-7a63-a7e7-655d7a515dec
6,Why do AI agents need tools?,None,"AttributeError(""'dict' object has no attribute...",Tools allow agents to perform tasks that an LL...,None,None,None,False,3.719462,ab4ec2d4-2700-401a-93a8-4740f0792b25,01a0beab-9662-7610-976f-98d2769b37b5
7,What is the difference between an AI agent and...,None,"AttributeError(""'dict' object has no attribute...",A chatbot primarily generates conversational r...,None,None,None,False,2.574203,b2a91cfe-72d1-420e-8605-b009b90643fe,01a0beab-acc6-7f92-b7d0-8395d91fb3b8
